# Exploratory Data Analysis (EDA)

**Dataset:** Heart Failure Clinical Records Dataset

This notebook performs a complete EDA workflow:
- Load and inspect the dataset
- Check data quality (types, missing values, duplicates)
- Summary statistics
- Univariate analysis (distributions)
- Bivariate analysis vs. the target (`DEATH_EVENT`)
- Correlation analysis
- Key insights & conclusions

> **Target column:** `DEATH_EVENT` (1 = death, 0 = survived during follow-up)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv(r"/mnt/data/heart_failure_clinical_records_dataset.csv")

# Basic preview
df.head()

## 1. Dataset overview

We first check the dataset size, column names, and data types to understand what we are working with.

In [ ]:
# Shape: (rows, columns)
df.shape

In [ ]:
# Column names
df.columns.tolist()

In [ ]:
# Data types and non-null counts
df.info()

### Column meaning (quick guide)

Typical meanings in this dataset:
- `age`: patient age (years)
- `anaemia`, `diabetes`, `high_blood_pressure`, `sex`, `smoking`: binary indicators (0/1)
- `creatinine_phosphokinase`: enzyme level (mcg/L)
- `ejection_fraction`: % of blood leaving the heart at each contraction
- `platelets`: platelets in blood (kiloplatelets/mL)
- `serum_creatinine`: level of serum creatinine (mg/dL)
- `serum_sodium`: level of serum sodium (mEq/L)
- `time`: follow-up period (days)
- `DEATH_EVENT`: target label (1 = death, 0 = survived)

*(If your instructor expects a dataset description section, you can paste the dataset source/description here.)*

## 2. Data quality checks

We check:
- Missing values
- Duplicate rows
- Basic validity (binary columns are only 0/1)

In [ ]:
# Missing values per column
df.isna().sum()

In [ ]:
# Number of duplicate rows
df.duplicated().sum()

In [ ]:
# Check binary columns contain only 0/1
binary_cols = ["anaemia","diabetes","high_blood_pressure","sex","smoking","DEATH_EVENT"]
{col: sorted(df[col].unique().tolist()) for col in binary_cols}

**Observation:** This dataset has no missing values and no duplicates. Binary columns are clean (0/1).

## 3. Summary statistics

We compute descriptive statistics to understand central tendency and spread. For binary columns, we also look at counts.

In [ ]:
# Summary for numeric columns
df.describe().T

In [ ]:
# Value counts for binary columns
for col in ["anaemia","diabetes","high_blood_pressure","sex","smoking","DEATH_EVENT"]:
    print(f"\n{col} value counts:")
    print(df[col].value_counts().sort_index())

### Target balance

Understanding how balanced the target is helps interpret model performance later.

In [ ]:
death_counts = df["DEATH_EVENT"].value_counts().sort_index()
death_counts, death_counts / len(df)

In [ ]:
# Plot target distribution
plt.figure()
plt.bar(["Survived (0)","Death (1)"], death_counts.values)
plt.title("Target distribution: DEATH_EVENT")
plt.ylabel("Count")
plt.show()

## 4. Univariate analysis (single-variable)

We visualize distributions for continuous variables and count plots for binary variables.

**Goal:** detect skewness, outliers, and typical ranges.

In [ ]:
# Split columns by type for plotting
numeric_cols = ["age","creatinine_phosphokinase","ejection_fraction","platelets",
                "serum_creatinine","serum_sodium","time"]
binary_cols = ["anaemia","diabetes","high_blood_pressure","sex","smoking"]

numeric_cols, binary_cols

In [ ]:
# Histograms for numeric columns
for col in numeric_cols:
    plt.figure()
    plt.hist(df[col], bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# Bar charts for binary columns
for col in binary_cols:
    vc = df[col].value_counts().sort_index()
    plt.figure()
    plt.bar([f"{col}=0", f"{col}=1"], vc.values)
    plt.title(f"Counts of {col}")
    plt.ylabel("Count")
    plt.show()

### Notes you can mention (typical patterns)

- Many clinical variables can be skewed (e.g., `creatinine_phosphokinase`).
- `time` indicates how long the patient was followed; its distribution can affect interpretation.
- Outliers may exist in lab measures; we confirm with boxplots next.

## 5. Bivariate analysis vs. outcome (`DEATH_EVENT`)

Here we compare each feature between **survivors (0)** and **death events (1)**.

**Goal:** identify features that differ noticeably by outcome.

In [ ]:
# Boxplots of numeric variables by DEATH_EVENT
for col in numeric_cols:
    plt.figure()
    data0 = df[df["DEATH_EVENT"]==0][col]
    data1 = df[df["DEATH_EVENT"]==1][col]
    plt.boxplot([data0, data1], labels=["Survived (0)", "Death (1)"])
    plt.title(f"{col} by outcome")
    plt.ylabel(col)
    plt.show()

In [ ]:
# Outcome rates for binary variables
for col in binary_cols:
    rate = df.groupby(col)["DEATH_EVENT"].mean()
    count = df[col].value_counts().sort_index()
    print(f"\n{col}:")
    print("Counts:", count.to_dict())
    print("Death rate (mean DEATH_EVENT):", rate.to_dict())

In [ ]:
# Visualize death rate by binary variable
for col in binary_cols:
    rate = df.groupby(col)["DEATH_EVENT"].mean().sort_index()
    plt.figure()
    plt.bar([f"{col}=0", f"{col}=1"], rate.values)
    plt.title(f"Death rate by {col}")
    plt.ylabel("Death rate")
    plt.ylim(0, 1)
    plt.show()

## 6. Correlation analysis

Correlation helps identify linear relationships and potential multicollinearity.

> Note: Correlation does **not** imply causation—especially in medical datasets.

In [ ]:
# Correlation matrix
corr = df.corr(numeric_only=True)
corr

In [ ]:
# Correlation heatmap (matplotlib only)
plt.figure(figsize=(10,8))
plt.imshow(corr, aspect='auto')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()

### Top correlations with the target

We list the features most correlated (positively/negatively) with `DEATH_EVENT`.

In [ ]:
target_corr = corr["DEATH_EVENT"].sort_values()
target_corr

In [ ]:
# Show strongest absolute correlations (excluding the target itself)
top_abs = target_corr.drop("DEATH_EVENT").abs().sort_values(ascending=False).head(8)
top_abs

## 7. Key insights (write-up)

Below is a ready-to-submit style summary. You can adjust wording based on your plots.

**Data quality**
- No missing values and no duplicate rows.
- Binary features are correctly encoded as 0/1.

**Target balance**
- `DEATH_EVENT` shows the proportion of death events vs. survivors; this affects evaluation metrics.

**Univariate observations**
- Some clinical measures can be skewed and may include outliers.
- Follow-up time (`time`) varies between patients.

**Bivariate observations (vs outcome)**
- Features that show visibly different distributions between outcome classes are candidates for strong predictors.
- Binary features can be compared via death-rate plots.

**Correlation**
- The correlation table provides a quick signal of which variables move with the outcome, but it should be supported by the bivariate plots above.

## 8. Conclusion

This EDA explored the heart failure clinical records dataset and examined distributions, group differences by outcome, and correlations.

**Next steps (if this is part of a machine learning pipeline):**
- Split data into train/test
- Standardize numeric features
- Train baseline models (Logistic Regression, Random Forest)
- Evaluate using accuracy + precision/recall/F1 and ROC-AUC (especially if the target is imbalanced)

---
*End of notebook.*